# SARA · Images and gestures

For first setup, follow the [Linux installation guide](../README.md#install-on-linux) and run `make notebooks` to install the kernel.

1. Run `make ui` in the SARA folder and open the printed HTTPS URL on the phone.
2. Select **Video** on the phone and this project's **.venv** kernel in VS Code.
3. Run the cells below. Start the capture cell, then hold **Hold to capture** on the phone.

The notebook reads the gateway's JPEG preview through `/api/frame.jpg`, up to 960 × 720.
The app's normal recognition continues during capture. These experiments run in the notebook.

## Connect

Run this notebook on the gateway computer. `GATEWAY` is the data source address.
Monitoring uses the admin password, which defaults to `admin`.

In [ ]:
import json
import time
from getpass import getpass
from http.cookiejar import CookieJar
from pathlib import Path
from urllib.request import HTTPCookieProcessor, Request, build_opener

PROJECT = next(path for path in (Path.cwd(), *Path.cwd().parents)
               if (path / "gateway/server.py").exists())
GATEWAY = "http://localhost:8080"
connection = build_opener(HTTPCookieProcessor(CookieJar()))
password = getpass("Admin password (Enter for admin): ") or "admin"
login = Request(GATEWAY + "/api/admin/login",
                data=json.dumps({"password": password}).encode(),
                headers={"Content-Type": "application/json"})
with connection.open(login, timeout=10) as response:
    print("Connected to SARA")
password = None


def get(path):
    with connection.open(GATEWAY + path, timeout=10) as response:
        return response.read(), response.headers

## Collect an image

Run this cell, then hold the phone's capture button. Release after the image appears.
`image` is a PIL RGB image and stays in notebook memory for repeated experiments.
`get_frame()` reads the latest live frame whenever you need another one.

In [ ]:
from io import BytesIO
from PIL import Image
from IPython.display import clear_output, display


def get_frame():
    data, _ = get("/api/frame.jpg")
    return Image.open(BytesIO(data)).convert("RGB") if data else None


def capture_frame(wait_seconds=15):
    print("Waiting for video. Hold the capture button on the phone.")
    deadline = time.monotonic() + wait_seconds
    while time.monotonic() < deadline:
        image = get_frame()
        if image is not None:
            return image
        time.sleep(0.1)
    raise TimeoutError("Select Video on the phone, then run this cell and hold capture again.")


image = capture_frame()
display(image)

## Try gesture recognition

This example uses Google's pretrained MediaPipe Gesture Recognizer. The model file is reused
from `.local/models` and downloaded on first use if needed.

Edit `predict_gesture(image)` to try another model or your own gesture logic.
For array-based models, `np.asarray(image)` provides RGB pixels.
MediaPipe's `result.hand_landmarks` is also available inside the function for custom hand logic.
[Model documentation](https://developers.google.com/edge/mediapipe/solutions/vision/gesture_recognizer).

In [ ]:
import mediapipe as mp
import numpy as np
from urllib.request import urlretrieve

MODEL_PATH = PROJECT / ".local/models/gesture_recognizer.task"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
if not MODEL_PATH.exists():
    model_url = "https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task"
    urlretrieve(model_url, MODEL_PATH)

recognizer = mp.tasks.vision.GestureRecognizer.create_from_options(
    mp.tasks.vision.GestureRecognizerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=mp.tasks.vision.RunningMode.IMAGE,
    )
)


def predict_gesture(image):
    rgb = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.asarray(image))
    result = recognizer.recognize(rgb)
    return [(hand[0].category_name, round(hand[0].score, 3)) for hand in result.gestures]


predict_gesture(image)

## Try live predictions

Run this cell, then hold the phone's capture button. It runs for 10 seconds and calls the
same editable `predict_gesture()` function. Captured images remain in `frames` for later tests.
Rerun the cell for another capture. The phone limits each hold to 15 seconds.

In [ ]:
frames = []
deadline = time.monotonic() + 10
print("Waiting for video. Hold the capture button on the phone.")
while time.monotonic() < deadline:
    frame = get_frame()
    if frame is not None:
        frames.append(frame)
        prediction = predict_gesture(frame)
        clear_output(wait=True)
        display(frame)
        print(prediction)
    time.sleep(0.1)
print(f"Collected {len(frames)} frames")